In [13]:
import jams
import numpy as np
import os
import pretty_midi
from music21 import stream, note, tempo, meter, clef, articulations, expressions, spanner, interval
import random
np.int = int # deprecated np.int
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import math

In [14]:
midi_dir= '/data/akshaj/MusicAI/GuitarSet/MIDIAnnotations/'
midi_path = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

In [15]:
# Standard guitar tuning (MIDI pitch for each open string)
STANDARD_TUNING = {
    6: 40,  # E2 (low E)
    5: 45,  # A2
    4: 50,  # D3
    3: 55,  # G3
    2: 59,  # B3
    1: 64   # E4 (high e)
}

In [16]:
def midi_pitch_to_guitar_positions(midi_pitch, tuning=STANDARD_TUNING, max_fret=12):
    """
    Find all possible (string, fret) positions for a given MIDI pitch.
    
    Args:
        midi_pitch: MIDI note number (e.g., 60 = middle C)
        tuning: Dictionary of {string_number: open_string_midi_pitch}
        max_fret: Maximum fret to consider
    
    Returns:
        List of (string, fret) tuples, sorted by preference
    """
    positions = []
    
    for string_num in range(1, 7):  # Strings 1-6
        open_pitch = tuning[string_num]
        fret = midi_pitch - open_pitch
        
        # Valid if fret is 0-12 (or your max_fret)
        if 0 <= fret <= max_fret:
            positions.append((string_num, fret))
    
    # Sort by preference: middle strings first, lower frets preferred
    # This creates more natural fingering
    def position_score(pos):
        string, fret = pos
        # Prefer middle strings (3, 4) and lower frets
        string_penalty = abs(string - 3.5)  # Prefer strings 3-4
        fret_penalty = fret * 0.1  # Slight preference for lower frets
        return string_penalty + fret_penalty
    
    positions.sort(key=position_score)
    return positions

In [17]:
def choose_best_position(midi_pitch, previous_position=None, tuning=STANDARD_TUNING):
    """
    Choose the best string/fret position for a pitch.
    Takes into account the previous position to minimize hand movement.
    
    Args:
        midi_pitch: MIDI note number
        previous_position: Previous (string, fret) tuple, or None
        tuning: Guitar tuning
    
    Returns:
        (string, fret) tuple
    """
    positions = midi_pitch_to_guitar_positions(midi_pitch, tuning)
    
    if not positions:
        # Pitch out of range - use closest approximation
        # Find the closest string
        closest_string = min(tuning.keys(), 
                           key=lambda s: abs(tuning[s] - midi_pitch))
        fret = max(0, min(12, midi_pitch - tuning[closest_string]))
        return (closest_string, fret)
    
    if previous_position is None:
        # No previous position - use most natural position
        return positions[0]
    
    # Choose position closest to previous position (minimize hand movement)
    prev_string, prev_fret = previous_position
    
    def distance_score(pos):
        string, fret = pos
        string_distance = abs(string - prev_string)
        fret_distance = abs(fret - prev_fret)
        # Weight fret distance more (moving along frets is harder than changing strings)
        return fret_distance * 2 + string_distance
    
    return min(positions, key=distance_score)

In [18]:
def midi_to_jams_with_tablature(midi_path, tuning=STANDARD_TUNING):
    """
    Convert MIDI file to JAMS with intelligent tablature mapping.
    
    Args:
        midi_path: Path to MIDI file
        tuning: Guitar tuning dictionary
    
    Returns:
        JAMS object with both note_midi and tab_note annotations
    """
    # Load MIDI
    pm = pretty_midi.PrettyMIDI(midi_path)
    guitar_notes = pm.instruments[0].notes
    
    # Create JAMS
    jam = jams.JAMS()
    
    # Add note_midi annotation
    note_ann = jams.Annotation(namespace='note_midi')
    for n in guitar_notes:
        note_ann.append(
            time=n.start,
            duration=n.end - n.start,
            value=n.pitch,
            confidence=n.velocity / 127
        )
    jam.annotations.append(note_ann)
    
    # Add tab_note annotation with intelligent string/fret mapping
    tab_ann = jams.Annotation(namespace='tab_note')
    
    previous_position = None
    for n in guitar_notes:
        # Find best position for this pitch
        string, fret = choose_best_position(n.pitch, previous_position, tuning)
        previous_position = (string, fret)
        
        value = {
            "pitch": n.pitch,
            "string": string,
            "fret": fret,
            "techniques": []  # No techniques from MIDI
        }
        
        tab_ann.append(
            time=n.start,
            duration=n.end - n.start,
            value=value,
            confidence=n.velocity / 127
        )
    
    jam.annotations.append(tab_ann)
    
    print(f"Converted {len(guitar_notes)} notes to tablature")
    print(f"Pitch range: {min(n.pitch for n in guitar_notes)} - {max(n.pitch for n in guitar_notes)}")
    
    return jam

In [19]:
def jams_to_musicxml_real(jam, output_xml='output.xml', tempo_bpm=120):
    """
    Convert JAMS with real tablature to MusicXML.
    Uses actual pitch information for correct notation.
    
    Args:
        jam: JAMS object with tab_note annotation
        output_xml: Output MusicXML file path
        tempo_bpm: Tempo in BPM
    
    Returns:
        Path to created XML file
    """
    # Find tab_note annotation
    tab_notes = None
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            tab_notes = ann
            break
    
    if tab_notes is None:
        raise ValueError("No tab_note annotation found")
    
    print(f"Converting {len(tab_notes.data)} notes to MusicXML...")
    
    # Create music21 score
    score = stream.Score()
    part = stream.Part()
    
    # Add metadata
    part.insert(0, tempo.MetronomeMark(number=tempo_bpm))
    part.insert(0, meter.TimeSignature('4/4'))
    part.insert(0, clef.TabClef())
    
    # Beat duration
    beat_sec = 60.0 / tempo_bpm
    
    # Convert notes
    previous_note = None
    for obs in tab_notes.data:
        val = obs.value
        
        # Use actual pitch from MIDI
        midi_pitch = val['pitch']
        string_num = val['string']
        fret_num = val['fret']

        # Extract techniques
        techniques = val.get('techniques', [])  # Extract techniques from JAMS
        
        # Create note with correct pitch
        n = note.Note()
        n.pitch.midi = midi_pitch

        # Apply techniques to note
        if techniques:
            print(f"  Processing {len(techniques)} technique(s)")
            for tech in techniques:
                if not tech:  # Skip None or empty values
                    print(f"  Skipping empty technique: {tech}")  # ← More specific
                    continue
                
                # Map guitar techniques to music21 notation
                if tech in ['hammer-on', 'hammer_on']:
                    print("hammer-on detected")
                    n.expressions.append(expressions.TextExpression('H'))
                    
                    # Create slur from previous note to this one
                    if previous_note is not None:
                        slur = spanner.Slur(previous_note, n)
                        part.insert(0, slur)  # Add slur to the part
            
                # PULL-OFF: Same as hammer-on
                elif tech in ['pull-off', 'pull_off']:
                    print("pull off detected")
                    n.expressions.append(expressions.TextExpression('P'))
                    
                    if previous_note is not None:
                        slur = spanner.Slur(previous_note, n)
                        part.insert(0, slur)
                
                elif tech == 'bend':
                    print("bend detected")
                    bend_amount = 0.5  # Default half step
                    # Create proper MusicXML bend element
                    bend = articulations.FretBend(
                        bendAlter=interval.Interval(bend_amount)
                    )
                    n.articulations.append(bend)
                    n.expressions.append(expressions.TextExpression('B'))
                
                elif tech == 'slide':
                    print("slide detected")
                    n.expressions.append(expressions.TextExpression('/'))
                
                elif tech == 'vibrato':
                    print("Vibrato detected")
                    n.expressions.append(expressions.TextExpression('~'))
                
                elif tech == 'harmonic':
                    print("Harmonic detected")
                    n.articulations.append(articulations.Harmonic())
        
        # Quantize duration to standard values
        duration_beats = obs.duration / beat_sec
        # Round to nearest eighth note
        quantized = round(duration_beats * 2) / 2
        if quantized < 0.5:
            quantized = 0.5  # Minimum eighth note
        n.quarterLength = quantized
        
        # Add tablature info
        n.editorial.stringNumber = string_num
        n.editorial.fretNumber = fret_num
        
        # Quantize time position
        time_beats = obs.time / beat_sec
        quantized_time = round(time_beats * 2) / 2
        
        part.insert(quantized_time, n)
        previous_note = n
    
    # Create measures
    part.makeMeasures(inPlace=True)
    score.append(part)
    
    # Write to MusicXML
    try:
        score.write('musicxml', fp=output_xml)
        print(f"✓ Created {output_xml}")
    except Exception as e:
        print(f"Error: {e}")
        print("Trying with makeNotation...")
        score.write('musicxml', fp=output_xml, makeNotation=True)
        print(f"✓ Created {output_xml} (with notation fixes)")
    
    return output_xml

In [20]:
# =============================================================================
# ADVANCED: Position optimization for better fingering
# =============================================================================

def optimize_tablature_positions(jam, max_stretch=4):
    """
    Optimize string/fret positions for better playability.
    Tries to keep notes within reasonable hand positions.
    
    Args:
        jam: JAMS object with tab_note annotation
        max_stretch: Maximum fret stretch (typically 4 frets)
    
    Returns:
        Modified JAMS object with optimized positions
    """
    # Find tab annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            tab_ann = ann
            break
    
    if not tab_ann or len(tab_ann.data) < 2:
        return jam
    
    print("Optimizing tablature positions for playability...")
    
    # Re-calculate positions considering hand position
    notes = list(tab_ann.data)
    current_position = None  # (center_fret, string)
    
    for i, obs in enumerate(notes):
        pitch = obs.value['pitch']
        
        # Get all possible positions
        positions = midi_pitch_to_guitar_positions(pitch)
        
        if not positions:
            continue
        
        if current_position is None:
            # First note - choose most natural position
            string, fret = positions[0]
            current_position = fret
        else:
            # Choose position within max_stretch of current position
            valid_positions = [
                (s, f) for s, f in positions
                if abs(f - current_position) <= max_stretch
            ]
            
            if valid_positions:
                # Choose closest to current position
                string, fret = min(valid_positions, 
                                  key=lambda p: abs(p[1] - current_position))
            else:
                # No position within reach - need to shift hand position
                string, fret = positions[0]
                current_position = fret
        
        # Update the note
        obs.value['string'] = string
        obs.value['fret'] = fret
    
    print("✓ Positions optimized")
    return jam

In [21]:
def add_random_techniques_to_existing_jam(jam, technique_probability=0.5):
    '''
    Adds random techniques to EXISTING tab_note annotation
    '''
    tech_options = ["slide", "vibrato", "hammer-on", "pull-off", "bend", "harmonic"]
    # Find the EXISTING tab_note annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == 'tab_note':
            tab_ann = ann
            break
    
    if tab_ann is None:
        raise ValueError("No tab_note annotation found! Run midi_to_jams_with_tablature() first")
    
    technique_count = 0
    
    # MODIFY existing notes (don't create new ones)
    for obs in tab_ann.data:
        # Add technique to existing note
        if random.random() < technique_probability:
            tech = random.choice(tech_options)
            obs.value['techniques'] = [tech]  # ← Modify existing
            technique_count += 1
        else:
            obs.value['techniques'] = []
    
    print(f"\n✓ Modified {len(tab_ann.data)} notes")
    print(f"✓ Added {technique_count} techniques ({technique_count/len(tab_ann.data)*100:.1f}%)")
    
    return jam

In [22]:
# Step 1: Create JAMS with tablature
jam = midi_to_jams_with_tablature(midi_path)

# Step 2: Add techniques to EXISTING tab_note annotation
jam = add_random_techniques_to_existing_jam(jam, technique_probability=0.5)

# Step 3: Debug - check techniques are there
tab_ann = [a for a in jam.annotations if a.namespace == 'tab_note'][0]
print("\nNotes with techniques:")
for i, obs in enumerate(tab_ann.data[:20]):
    techs = obs.value.get('techniques', [])
    if techs:
        print(f"  Note {i}: {techs}")

# Step 4: Convert to MusicXML
jams_to_musicxml_real(jam, 'test_output', tempo_bpm=108)

Converted 61 notes to tablature
Pitch range: 56 - 75

✓ Modified 61 notes
✓ Added 29 techniques (47.5%)

Notes with techniques:
  Note 1: ['harmonic']
  Note 3: ['bend']
  Note 4: ['bend']
  Note 5: ['pull-off']
  Note 6: ['hammer-on']
  Note 7: ['harmonic']
  Note 8: ['vibrato']
  Note 9: ['bend']
  Note 10: ['slide']
  Note 12: ['slide']
  Note 13: ['harmonic']
  Note 15: ['harmonic']
Converting 61 notes to MusicXML...
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
bend detected
  Processing 1 technique(s)
bend detected
  Processing 1 technique(s)
pull off detected
  Processing 1 technique(s)
hammer-on detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Vibrato detected
  Processing 1 technique(s)
bend detected
  Processing 1 technique(s)
slide detected
  Processing 1 technique(s)
slide detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
bend detected

'test_output'